# EchoFactory - VALVE: Training STgram-MFN
**GPU**: Aktifkan T4 GPU di Kaggle Settings.


In [ ]:
import os
import gc
import json
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
MACHINE_TYPE = 'valve'
FEAT_DIR = '/kaggle/working/features'
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 5e-4
EMBED_DIM = 256
ARC_S, ARC_M = 32.0, 0.5
OUT_MODEL = f'/kaggle/working/stgram_mfn_valve.pt'


In [ ]:
class MIMIIDataset(Dataset):
    def __init__(self, machine, feat_dir, cond='normal'):
        path = os.path.join(feat_dir, f'{machine}_{cond}.pt')
        data = torch.load(path)
        self.feats = data['features']
        self.labels = data['labels']
        self.n_cls = int(self.labels.max().item()) + 1
    def __len__(self): return len(self.feats)
    def __getitem__(self, idx):
        f = self.feats[idx]
        return f[0:1], f[1:2], self.labels[idx]

train_ds = MIMIIDataset(MACHINE_TYPE, FEAT_DIR, 'normal')
N_CLASSES = train_ds.n_cls
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print(f'DataLoader: {len(train_dl)} batches | {N_CLASSES} machine IDs')


In [ ]:
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False), nn.BatchNorm2d(oc), nn.PReLU(oc))
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(ConvBNPReLU(ic, ic, s=s, g=ic), ConvBNPReLU(ic, oc, k=1, p=0))
    def forward(self, x): return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=256):
        super().__init__()
        self.enc = nn.Sequential(ConvBNPReLU(1, 32, s=2), DepthwiseSep(32, 64), DepthwiseSep(64, 128, s=2), DepthwiseSep(128, 128), DepthwiseSep(128, 256, s=2), DepthwiseSep(256, 256), DepthwiseSep(256, 512, s=2), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed))
    def forward(self, x): return self.head(self.enc(x))

class ArcFace(nn.Module):
    def __init__(self, ed, nc, s=32.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m
    def forward(self, feat, labels):
        cos = F.normalize(feat, 1) @ F.normalize(self.W, 1).T
        sin = (1.0 - cos ** 2 + 1e-8).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = F.one_hot(labels, cos.shape[1]).float()
        out = (one_hot * phi + (1.0 - one_hot) * cos) * self.s
        return F.cross_entropy(out, labels)

class STgramMFN(nn.Module):
    def __init__(self, nc, ed=256):
        super().__init__()
        self.mel, self.tgram = MobileFaceNet(ed), MobileFaceNet(ed)
        self.fuse = nn.Sequential(nn.Linear(ed * 2, ed), nn.BatchNorm1d(ed), nn.PReLU(ed))
        self.arc = ArcFace(ed, nc, ARC_S, ARC_M)
    def forward(self, mel, tg, labels=None):
        feat = F.normalize(self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1)), dim=1)
        if labels is not None: return feat, self.arc(feat, labels)
        return feat


In [ ]:
model = STgramMFN(N_CLASSES, EMBED_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
scaler = GradScaler()

best_loss = float('inf')
losses = []
t0 = time.time()

print(f'Training {MACHINE_TYPE.upper()} | {EPOCHS} epochs')
for ep in range(1, EPOCHS + 1):
    model.train(); epoch_loss = 0.0
    for mel, tg, lab in train_dl:
        mel, tg, lab = mel.to(device), tg.to(device), lab.to(device)
        optimizer.zero_grad()
        with autocast(): _, loss = model(mel, tg, lab)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    avg = epoch_loss / len(train_dl)
    losses.append(avg); scheduler.step()
    if avg < best_loss:
        best_loss = avg
        torch.save({'epoch': ep, 'model_state': model.state_dict(), 'best_loss': best_loss, 'machine': MACHINE_TYPE, 'n_classes': N_CLASSES, 'embed_dim': EMBED_DIM}, OUT_MODEL)
    if ep % 10 == 0 or ep == 1:
        print(f'Epoch {ep:2d}/{EPOCHS} | Loss: {avg:.4f}')
